로깅

In [ ]:
import logging

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s %(levelname)-8s %(name)s %(message)s",
    datefmt="%H:%M:%S", # 시:분:초
    force=True
)

# 이름을 붙여서 로거를 하나 가져온다.
log = logging.getLogger("demo")

log.debug("디버그 모드의 debug입니다,")
log.info("인포레벨 로그입니다."),
log.warning("경고 경고 주의하세요.")
log.error("에러 발생!!")
# 로그 포맷 패턴
# %(asctime)s : 시간
# %(levelname)s : 로그레벨
# %(name) : 로거 이름
# %(message) : 메시지

13:19:34 DEBUG    demo 디버그 모드의 debug입니다,
13:19:34 INFO     demo 인포레벨 로그입니다.
13:19:34 WARNING  demo 경고 경고 주의하세요.
13:19:34 ERROR    demo 에러 발생!!


In [ ]:
import sys, logging

if "backend" not in sys.path:
    sys.path.insert(0, "backend")


# 앞 셀에서 루트 로거를 건드렸으면, 아래와 같이 깨끗이 하고 새로 얹는다.
logging.getLogger().handlers = []
for m in [k for k in list(sys.modules) if k.startswith("app.core.logging")]:
    del sys.modules[m]

# ----------------------------------------------------------------

from app.core.logging import get_logger

logger = get_logger("app.services.document") # 임시 패키지명
logger.info("문서 적재 완료: %s", "DOC-HR-001")  # f"" -> %s 권장
logger.warning("임계치에 근접합니다. 주의하세요.")

로깅 활용 예시

In [ ]:
class AgentError(Exception):
    status_code, code = 400, "agent_error"
    def __init__(self, message, *, detail=None):
        super().__init__(message)
        self.message, self.detail = message, detail

class NotFound(AgentError):
    status_code, code = 404, "not_found"

class ExternalServiceError(AgentError):
    status_code, code = 502, "external_error"

def fake_parsing(doc_id):
    if doc_id == "DOC-BAD":
        raise ConnectionError("Connection refused: api.llm.ai:443")
    return f"{doc_id}의 본문 텍스트"

def ingest(doc_id):
    logger.info("적재 시작: %s", doc_id)
    try:
        text = fake_parsing(doc_id)
    except ConnectionError as e:
        logger.exception("문서 파싱 실패 : %s", doc_id)
        # 우리 타입의 예외로 변환해 발생 시키기
        raise ExternalServiceError("문서 변환 서비스에 연결하지 못했습니다.", detail=str(e)) from e

    logger.info("적재 완료: %s", doc_id)
    return text


# 문제없는 상황
ingest("DOC-HR-001")
# 예외 발생 상황
try:
    ingest("DOC-BAD")
except AgentError as e:
    print("Agent Error")
    print(e.status_code)
    print(e.code)
    print(e.message)